Number of Crossings
=
This notebook finds every point where two line segments on **different** layers cross (e.g. LV trenches, grounding trenches and fences), counts them, and writes an AutoCAD script that draws a circle at each crossing so they can be checked and tagged.

**Workflow overview:**
1. Read the line segments (start/end points and layer) from an Excel export.
2. Build a spatial index and test each segment only against the nearby ones.
3. Keep the point intersections between segments of different layers (the number of rows is the number of crossings).
4. Write `Cruzamentos.scr` with one `CIRCLE` per crossing.

**Input prerequisites** (the code assumes all of these, and it doesn't check them):
- **File:** `C:\Users\Usuario\Desktop\teste.xls` (hardcoded; `.xls` needs the `xlrd` package). It is an AutoCAD data extraction of the lines of the layers to compare.
- **Columns:** `Start X`, `Start Y`, `End X`, `End Y`, `Layer`, one row per straight segment.
  - Polylines must be exploded into lines first; arcs are not handled.
- **Layers:** crossings are only counted between different `Layer` values, compared as exact, case-sensitive strings. Two lines on the same layer never count.

**AutoCAD:** run `Cruzamentos.scr` in the drawing to draw a circle of radius 10 at each crossing.

Loading the lines
=
Reads the extracted segments into `df_lines` and prints them. The `OLE2 inconsistency` warning comes from the `.xls` reader and is harmless.

`numpy` is imported but not used.

In [5]:
import pandas as pd
import numpy as np
from shapely.geometry import LineString
from shapely.strtree import STRtree

df_lines = pd.read_excel(r"C:\Users\Usuario\Desktop\teste.xls")
print(df_lines)

WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
           End X        End Y     Start X      Start Y                Layer
0     48469.3696  156111.0344  48486.1426  156128.2003  EMF_civ_vedacao_R1b
1     48469.3696  156087.0344  48469.3696  156111.0344  EMF_civ_vedacao_R1b
2     48483.3250  156208.6981  48494.1215  156251.3530  EMF_civ_vedacao_R1b
3     48486.1426  156128.2003  48507.0738  156139.9426  EMF_civ_vedacao_R1b
4     48494.1215  156251.3530  48538.3463  156285.7066  EMF_civ_vedacao_R1b
...          ...          ...         ...          ...                  ...
6415  50321.4685  155824.8372  50319.9340  155834.0588  EMF_civ_vedacao_R1b
6416  50322.7754  154982.3403  50293.8060  155024.7650  EMF_civ_vedacao_R1b
6417  50326.3528  155900.2080  50329.3260  155923.6687  EMF_civ_vedacao_R1b
6418  50327.3831  155959.2248  50291.7315  155959.2248  EMF_civ_vedacao_R1b
6419  50329.3260  155923.6687  50327.3831  155959.2248  EMF_civ_vedacao_R1b

[6420 rows x 5

Finding the crossings
=
`find_intersections(df)`:
- Builds a shapely `LineString` per row and keeps its layer in a parallel list.
- `STRtree` is a spatial index. `tree.query(geom)` returns the indices of the segments whose bounding boxes touch it. Returning indices is the shapely 2.x behaviour; in 1.x it returns geometries and this code breaks.
- `i < j` tests each pair only once. Pairs on the same layer are skipped.
- Only intersections of type `Point` are kept. Overlapping collinear segments (a `LineString` result) and multi-point results are ignored.
- **Warning:** a crossing exactly at a vertex shared by two consecutive segments of the same line is found once per segment. The results are not de-duplicated, so the count can be inflated (e.g. rows 3 and 4 of the saved output have the same point).

Output: `results` (layer 1, layer 2, x, y; its row count is the number of crossings). It then writes `C:\Users\Usuario\Desktop\Pessoal\Python\Tagueamento Cruzamentos\Cruzamentos.scr`, with one `_.CIRCLE x,y 10` (radius 10) per crossing.

In [6]:
def find_intersections(df):
    """
    Simplified version without complex index handling
    Excludes comparisons between lines with the same Layer
    """
    # Create LineString objects
    geometries = []
    layers = []  # Store the layer information for each geometry

    for _, row in df.iterrows():
        geom = LineString([(row['Start X'], row['Start Y']),
                         (row['End X'], row['End Y'])])
        geometries.append(geom)
        layers.append(row['Layer'])  # Store the layer

    # Build spatial index
    tree = STRtree(geometries)

    intersections = []
    n = len(geometries)

    for i in range(n):
        # Query returns indices of potentially intersecting geometries
        candidate_indices = tree.query(geometries[i])

        for j in candidate_indices:
            # Skip if same layer or same line
            if i < j and layers[i] != layers[j]:
                if geometries[i].intersects(geometries[j]):
                    intersection_point = geometries[i].intersection(geometries[j])

                    if intersection_point.geom_type == 'Point':
                        intersections.append({
                            'line1_layer': layers[i],
                            'line2_layer': layers[j],
                            'x': intersection_point.x,
                            'y': intersection_point.y
                        })

    return pd.DataFrame(intersections)

# Usage
results = find_intersections(df_lines)
print(results)

with open(r"C:\Users\Usuario\Desktop\Pessoal\Python\Tagueamento Cruzamentos\Cruzamentos.scr", 'w') as f:
    for _, row in results.iterrows():
        x, y = row['x'], row['y']
        f.write(f"_.CIRCLE {x:.3f},{y:.3f} 10\n")



                         line1_layer                     line2_layer  \
0                  EMF_bt_vala_solar                       emf_bt_ca   
1     EMF_vala_aterramento_exclusiva                       emf_bt_ca   
2                  EMF_bt_vala_solar                       emf_bt_ca   
3                  EMF_bt_vala_solar  EMF_vala_aterramento_exclusiva   
4                  EMF_bt_vala_solar  EMF_vala_aterramento_exclusiva   
...                              ...                             ...   
1031                       emf_bt_ca             EMF_civ_vedacao_R1b   
1032                       emf_bt_ca  EMF_vala_aterramento_exclusiva   
1033  EMF_vala_aterramento_exclusiva               EMF_bt_vala_solar   
1034  EMF_vala_aterramento_exclusiva                       emf_bt_ca   
1035                       emf_bt_ca  EMF_vala_aterramento_exclusiva   

                 x              y  
0     48616.860000  156325.400200  
1     48641.471981  156319.673463  
2     48695.499306  156296.

Empty cell (unused).